# 04 — Test Run & Demo GIFs

Runs the trained YOLOv8 model on qasim21 frames grouped into 3 visual clusters and exports 3 annotated GIFs (one per cluster) for the dashboard.

| | |
|---|---|
| **Model** | YOLOv8s fine-tuned on merged person dataset |
| **Data** | qasim21 video stream frames (3 clusters by visual similarity) |
| **Output** | `demo_detection_1.gif`, `demo_detection_2.gif`, `demo_detection_3.gif` (~15 frames each) |

In [ ]:
!pip install ultralytics kaggle pyyaml opencv-python-headless matplotlib pillow imageio scikit-learn -q

import os, shutil, glob, random, json, re
import cv2
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image
import imageio
import ultralytics
from ultralytics import YOLO

print('Ultralytics:', ultralytics.__version__)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

DRIVE_ROOT = '/content/drive/MyDrive/AI_TRAINING/GreenVision'
os.makedirs(DRIVE_ROOT, exist_ok=True)

## 1. Setup Model

In [ ]:
import torch

CONF = 0.5
DEVICE = 0 if torch.cuda.is_available() else 'cpu'

# Find best.pt
LOCAL_BEST = '/content/best.pt'
candidates = [os.path.join(DRIVE_ROOT, 'models', 'best.pt')]
runs_dir = os.path.join(DRIVE_ROOT, 'runs')
if os.path.exists(runs_dir):
    for rd in sorted(glob.glob(os.path.join(runs_dir, 'person_detect_v1*'))):
        candidates.append(os.path.join(rd, 'weights', 'best.pt'))

DRIVE_BEST = None
for c in candidates:
    if os.path.exists(c):
        DRIVE_BEST = c
        break

if DRIVE_BEST is None:
    raise FileNotFoundError('best.pt not found. Run 02_train.ipynb first.')

if not os.path.exists(LOCAL_BEST):
    shutil.copy2(DRIVE_BEST, LOCAL_BEST)

model = YOLO(LOCAL_BEST)
print(f'Model: {DRIVE_BEST}')
print(f'Device: {DEVICE}')

## 2. Get qasim21 Images

In [ ]:
from google.colab import userdata

kaggle_dir = os.path.expanduser('~/.kaggle')
os.makedirs(kaggle_dir, exist_ok=True)
token = userdata.get('KAGGLE_API_TOKEN')
username = 'dngdngphmminh'
creds = {"username": username, "key": token}
with open(os.path.join(kaggle_dir, 'kaggle.json'), 'w') as f:
    json.dump(creds, f)
os.chmod(os.path.join(kaggle_dir, 'kaggle.json'), 0o600)
print('Kaggle ready.')

In [ ]:
DRIVE_QASIM = os.path.join(DRIVE_ROOT, 'datasets', 'qasim21', 'images')

if os.path.exists(DRIVE_QASIM):
    all_images = glob.glob(os.path.join(DRIVE_QASIM, '*.jpg')) + glob.glob(os.path.join(DRIVE_QASIM, '*.png'))
    originals = [p for p in all_images if '_flip' not in os.path.basename(p) and '_bc' not in os.path.basename(p)]
    print(f'Found {len(originals)} original images on Drive')
    IMAGE_POOL = originals
else:
    print('Downloading qasim21...')
    !kaggle datasets download -d qasim21/video-stream-dataset-for-yolov5v7-for-detection -p /content --unzip -q
    all_images = glob.glob('/content/**/*.jpg', recursive=True) + glob.glob('/content/**/*.png', recursive=True)
    all_images = [p for p in all_images if '/content/drive' not in p and '/content/sample_data' not in p]
    print(f'Found {len(all_images)} images')
    IMAGE_POOL = all_images

if len(IMAGE_POOL) == 0:
    raise FileNotFoundError('No images found.')

## 3. Group by Similarity, Select Frames from All 3 Clusters, Run Inference

In [ ]:
from sklearn.cluster import KMeans

# Compute color histograms for all images (fast similarity proxy)
print(f'Computing features for {len(IMAGE_POOL)} images...')

features = []
valid_images = []

for img_path in IMAGE_POOL:
    img = cv2.imread(img_path)
    if img is None:
        continue
    img_small = cv2.resize(img, (64, 64))
    hsv = cv2.cvtColor(img_small, cv2.COLOR_BGR2HSV)
    hist_h = cv2.calcHist([hsv], [0], None, [32], [0, 180]).flatten()
    hist_s = cv2.calcHist([hsv], [1], None, [32], [0, 256]).flatten()
    hist_v = cv2.calcHist([hsv], [2], None, [32], [0, 256]).flatten()
    feat = np.concatenate([hist_h, hist_s, hist_v])
    feat /= (feat.sum() + 1e-7)
    features.append(feat)
    valid_images.append(img_path)

features = np.array(features)
print(f'Features shape: {features.shape}')

# Cluster into 3 groups
kmeans = KMeans(n_clusters=3, random_state=42, n_init=10)
labels = kmeans.fit_predict(features)

cluster_sizes = [int(np.sum(labels == i)) for i in range(3)]
print(f'Cluster sizes: {cluster_sizes}')

# Sort images by filename number for consecutive frame selection
def extract_number(path):
    name = os.path.splitext(os.path.basename(path))[0]
    nums = re.findall(r'\d+', name)
    return int(nums[-1]) if nums else 0

GREEN = (0, 255, 0)
RED = (0, 0, 255)
FONT = cv2.FONT_HERSHEY_SIMPLEX

cluster_frames = {}  # cluster_id -> list of RGB frames

for cluster_id in range(3):
    indices = np.where(labels == cluster_id)[0]
    cluster_images = sorted([valid_images[i] for i in indices], key=extract_number)
    n = len(cluster_images)

    # Take up to 15 consecutive frames from the middle
    take = min(15, n)
    start = max(0, n // 2 - take // 2)
    selected = cluster_images[start:start + take]
    print(f'\nCluster {cluster_id}: {n} images -> selected {len(selected)} frames')

    frames = []
    for i, img_path in enumerate(selected):
        img = cv2.imread(img_path)
        if img is None:
            continue

        h, w = img.shape[:2]
        img = cv2.resize(img, (w // 2, h // 2))

        result = model.predict(img, conf=CONF, classes=[0], verbose=False, device=DEVICE)[0]

        person_count = 0
        if result.boxes is not None and len(result.boxes) > 0:
            person_count = len(result.boxes)
            for box in result.boxes:
                x1, y1, x2, y2 = map(int, box.xyxy[0].tolist())
                conf = float(box.conf[0])
                cv2.rectangle(img, (x1, y1), (x2, y2), GREEN, 2)
                cv2.putText(img, f'person {conf:.2f}', (x1, y1 - 8), FONT, 0.5, GREEN, 2)

        cv2.putText(img, f'People: {person_count}', (10, 30), FONT, 0.9, RED, 2)
        status = 'Reasonable' if person_count > 0 else 'Empty'
        cv2.putText(img, status, (10, 60), FONT, 0.7, GREEN, 2)

        frames.append(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))
        print(f'  Frame {i+1}: {person_count} people, size {img.shape[1]}x{img.shape[0]}')

    cluster_frames[cluster_id] = frames
    print(f'Cluster {cluster_id} done: {len(frames)} frames')

## 4. Export 3 GIFs

In [ ]:
OUTPUT_DIR = '/content/demo_outputs'
DRIVE_OUTPUT = os.path.join(DRIVE_ROOT, 'demo_outputs')
os.makedirs(OUTPUT_DIR, exist_ok=True)
os.makedirs(DRIVE_OUTPUT, exist_ok=True)

for cluster_id, frames in cluster_frames.items():
    if len(frames) < 2:
        print(f'Cluster {cluster_id}: skipping (only {len(frames)} frames)')
        continue

    gif_num = cluster_id + 1  # 1-indexed: 1, 2, 3
    fname = f'demo_detection_{gif_num}.gif'
    local_path = os.path.join(OUTPUT_DIR, fname)
    drive_path = os.path.join(DRIVE_OUTPUT, fname)

    imageio.mimsave(local_path, frames, format='GIF', duration=500, loop=0)

    # Verify
    check = Image.open(local_path)
    animated = getattr(check, 'is_animated', False)
    n_frames = getattr(check, 'n_frames', 1)
    size_kb = os.path.getsize(local_path) / 1024

    print(f'Cluster {cluster_id} -> GIF {gif_num}: {fname}')
    print(f'  Size: {check.size}, Animated: {animated}, Frames: {n_frames}, File: {size_kb:.0f} KB')

    assert n_frames > 1, f'{fname} has only 1 frame!'

    shutil.copy2(local_path, drive_path)
    print(f'  Saved to Drive: {drive_path}')

print('\nAll 3 GIFs exported!')

## 5. Preview All 3 GIFs

In [ ]:
fig, axes = plt.subplots(3, 1, figsize=(18, 12))
for ax, (cluster_id, frames) in zip(axes, sorted(cluster_frames.items())):
    if len(frames) < 2:
        continue
    # Show 5 sample frames side by side
    indices = [0, len(frames)//4, len(frames)//2, 3*len(frames)//4, -1]
    sample = np.concatenate([frames[i] for i in indices], axis=1)
    ax.imshow(sample)
    ax.axis('off')
    ax.set_title(f'Cluster {cluster_id} ({len(frames)} frames)', fontsize=12, fontweight='bold')
plt.suptitle('All 3 Cluster GIF Previews', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
from IPython.display import display, Image as IPImage, HTML

for gif_num in range(1, 4):
    path = os.path.join(OUTPUT_DIR, f'demo_detection_{gif_num}.gif')
    if os.path.exists(path):
        display(HTML(f'<h3>GIF {gif_num}</h3>'))
        with open(path, 'rb') as f:
            display(IPImage(data=f.read(), format='gif'))
    else:
        print(f'GIF {gif_num} not found')

---
## Done!

3 GIFs saved to Drive:
- `/content/drive/MyDrive/AI_TRAINING/GreenVision/demo_outputs/demo_detection_1.gif`
- `/content/drive/MyDrive/AI_TRAINING/GreenVision/demo_outputs/demo_detection_2.gif`
- `/content/drive/MyDrive/AI_TRAINING/GreenVision/demo_outputs/demo_detection_3.gif`

Download and place in `frontend/public/` for the dashboard.